# Setup (No Training Here)


## Imports

In [ ]:
using ComponentArrays
using Random
using LinearAlgebra
using SparseArrays
using LinearSolve
using OrdinaryDiffEq
using OrdinaryDiffEqSDIRK
using OrdinaryDiffEqLowOrderRK
using SciMLSensitivity
using ADTypes
using Zygote
using Enzyme
using Optimization
using OptimizationOptimisers
using OptimizationOptimJL
using LineSearches
using Lux
using Functors
using Plots

# Enzyme.API.runtimeActivity!(true)
Enzyme.set_runtime_activity(Enzyme.Reverse)


# ### ADJUSTED: Load local training helpers from Research_Code/src.
include(joinpath(@__DIR__, "..", "..", "src", "Misc.", "misc_helpers.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "Local", "integration_AC.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "Local", "FOM_opt_AC.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "Local", "ROM_opt_AC.jl"))

## Forward Solve

In [ ]:
# parameters
N = 256
L = 1.0
Δx = L/(N+1)
x = L*Δx*collect(1:N)

ε2 = 1e-2
k = 1.0
p0 = ComponentVector(ε2=ε2, k=k, Δx=Δx)
u0 = tanh.((x .- L/2) / sqrt(2ε2))

tspan = (0.0, 2.0)
@show Δt = 0.5 * Δx^2/(2ε2)
t = LinRange(tspan[1], tspan[2], floor(Int,(tspan[2]-tspan[1])/Δt))

#f = ODEFunction(rhs_ac!)
# jac_prototype builts a prototype matrix for the jacobian of the system
# in this case, tridiagonal; the free energy term is pointwise, and the laplacian is tridiagonal
# but do we even need the jacobian if we're using an Euler solver?? idts. mainly for implicit (eg TRBDF2) solver
f = ODEFunction(rhs_ac!; jac_prototype=Tridiagonal(zeros(N-1),zeros(N),zeros(N-1)))

prob = ODEProblem(f, u0, tspan, p0)

alg = Euler()
# set up an adjoint solver. autojacvec will help the solver differentiate through the RHS
# note that this isn't just reverse mode AD. Gauss has to build an adjoint of the true rhs so that it can integrate however it needs to
# however, in this case, the RHS is discrete (discrete laplacian + FE eval)
sensalg = GaussAdjoint(autojacvec=EnzymeVJP())

# solve the forwards problem
@time u_ref = solve(prob, alg; dt=Δt, saveat=t)
@time u_ref = solve(prob, alg; dt=Δt, saveat=t);


In [ ]:
@gif for i in 1:length(u_ref.t)
    plot(x, u_ref.u[i], lw=2, ylim=(-1,1), label="")
end every 64

## Neural PDE Training

In [ ]:
f_true(u) = u^3 - u
plot(u -> f_true(u), xlim=(-1,1), ylim=(-0.5,0.5), lw=2, label="True", title="Learned vs true FE derivatives")

In [ ]:
A = get_lap1d_matrix(N,ε2,Δx)
prob_ROM = prepare_ROM_optimization(A, u_ref, 20, 20; N_obs=100)

In [ ]:
optprob_ROM = set_up_ROM_optimization(prob_ROM)

In [ ]:
output = run_ROM_optimization(
    optprob_ROM;
    η=[1e-3, 1e-4, 1e-5],
    β=(0.9, 0.99),
    N_iter=[10, 20, 30],
    warmup=true,
    save_frequency=10,
)

In [ ]:
save_dir = save_ROM_optimization_data(output, run_name)

## Visualize 

In [ ]:
# LOAD DATA
using Serialization

# ### ADJUSTED: Resolve saved runs from Optimization/Data after moving this notebook.
experiment_dir = joinpath(@__DIR__, "..", "Data", "ROM_AC_training_011_06_25_2026_100_obs_points_20_modes")
rom_data = deserialize(joinpath(experiment_dir, "rom_data.jls"))
parameter_history = deserialize(joinpath(experiment_dir, "parameter_history.jls"))
evaluation_history = deserialize(joinpath(experiment_dir, "evaluation_history.jls"))

output = (;
    result=(; u=parameter_history[end].θ),
    parameter_history,
    evaluation_history,
    final_loss=parameter_history[end].loss,
)


In [ ]:
pwd()

In [ ]:
println("η = [0.001, 0.0001, 1.0e-5]")

rng = MersenneTwister(rom_data.seed)
nn = Chain(Dense(1 => rom_data.h, tanh), Dense(rom_data.h => rom_data.h, tanh), Dense(rom_data.h => 1))
ps₀, state = Lux.setup(rng, nn)
ps₀ = fmap(x -> Float64.(x), ps₀)
_, re = Optimisers.destructure(ps₀)
ps_opt = re(output.result.u)

us = range(-1, 1; length=200)
Fvals = [Fnn(u, nn, ps_opt, state) for u in us]

plot(
    u -> rom_data.k * (u^3 - u);
    xlim=(-1, 1),
    ylim=(-0.7, 0.7),
    lw=2,
    label="True RHS nonlinearity",
    title = "True vs. Learned f(u)\nTrained on ROM",
)

plot!(
    us,
    -Fvals;
    lw=2,
    label="Neural network",
    ls=:dash,
)

# savefig("../../talks_and_papers/Spark talk/rom_nn_function_approximation.png")


In [ ]:
iterations = getproperty.(evaluation_history, :iteration)
losses = getproperty.(evaluation_history, :loss)

loss_plot = plot(
    iterations,
    losses;
    xlabel="Optimization iteration",
    ylabel="Loss",
    title="Loss history",
    lw=2,
    marker=:circle,
    label=false,
    yscale=:log10,
)


plot(loss_plot; size=(800, 700))

In [ ]:
println("Final NN training loss = ", output.final_loss)
